In [1]:
import os, pickle, re
import numpy as np
import pandas as pd
import json

from physics.simulation import mcfm, msq
from physics.hstar import c6, eft
from physics.variables import eft_terms
from physics.constants import c6_to_cH
from nsbi import taylr

import torch
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

In [2]:
torch.set_float32_matmul_precision('high')

import matplotlib, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})
lw = 1.5

In [3]:
run_dir = 'run/'

In [4]:
events_4l, scaler_X_4l, models_4l, scalers_y_4l = taylr.utils.load_results(os.path.join(run_dir,'h4l'), eft_terms)
events_2l2v, scaler_X_2l2v, models_2l2v, scalers_y_2l2v = taylr.utils.load_results(os.path.join(run_dir,'h2l2v'), eft_terms)

In [5]:
features_4l = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy',
            'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy',
            'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy',
            'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']
features_2l2v = ["l1_pt", "l1_eta", "l1_phi", "l1_energy", "l2_pt", "l2_eta", "l2_phi", "l2_energy", "met", "met_phi"]

batch_size = 1024

lumi = 3000.0

c6_linspace = [-25,25,201]
c6_space = np.linspace(*c6_linspace)
c6_sm = 0.0
i_c6_sm = np.where(np.round(c6_space,4)==c6_sm)[0][0]
c6_val_asimov = 0.0
i_c6_asimov = np.where(np.round(c6_space,4)==c6_val_asimov)[0][0]

In [6]:
class ZeroWeightFilter():
    def __init__(self):
        pass
    def __call__(self, kinematics, components, weights, probabilities):
        return np.where(weights != 0)

events_4l = events_4l.sample(n=100_000)
events_2l2v = events_2l2v.filter(ZeroWeightFilter()).sample(n=100_000, random_state=42)

w_4l_sm = events_4l.weights.to_numpy()
w_2l2v_sm = events_2l2v.weights.to_numpy()
w_sm = np.concatenate((w_4l_sm, w_2l2v_sm))

sigma_4l_sm = np.sum(w_4l_sm)
sigma_2l2v_sm = np.sum(w_2l2v_sm)
sigma_sm = np.sum(w_sm)

In [7]:
c6_4l_mod = eft.Modifier(baseline=msq.Component.SBI, events=events_4l)
c6_2l2v_mod = eft.Modifier(baseline=msq.Component.SBI, events=events_2l2v)

w_4l_c6, p_4l_c6 = c6_4l_mod.modify(c6=c6_space, ct=None, cg=None)
w_2l2v_c6, p_2l2v_c6 = c6_2l2v_mod.modify(c6=c6_space, ct=None, cg=None)

coeffs_4l_true = c6_4l_mod.coefficients
coeffs_2l2v_true = c6_2l2v_mod.coefficients
coeffs_true = np.vstack((coeffs_4l_true, coeffs_2l2v_true))

In [8]:
coeffs_4l = taylr.utils.get_coefficients(events_4l, features_4l, scaler_X_4l, models_4l, scalers_y_4l)
coeffs_2l2v = taylr.utils.get_coefficients(events_2l2v, features_2l2v, scaler_X_2l2v, models_2l2v, scalers_y_2l2v)
coeffs = np.vstack((coeffs_4l, coeffs_2l2v))

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py

Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

...

Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

In [9]:
# 4l+2l2v
# true estimate
w_c6_true = w_sm[:,None] * eft.msq_eft_over_sm(coeffs_true, c6_space)
sigma_c6_true = np.sum(w_c6_true, axis=0)
# NSBI estimate
w_c6 = w_sm[:,None] * eft.msq_eft_over_sm(coeffs, c6_space)
sigma_c6 = np.sum(w_c6, axis=0)

# 4l
# true estimate
w_4l_c6_true = w_4l_sm[:,None] * eft.msq_eft_over_sm(coeffs_4l_true, c6_space)
sigma_4l_c6_true = np.sum(w_4l_c6_true, axis=0)
# NSBI estimate
w_4l_c6 = w_4l_sm[:,None] * eft.msq_eft_over_sm(coeffs_4l, c6_space)
sigma_4l_c6 = np.sum(w_4l_c6, axis=0)

# 2l2v
# true estimate
w_2l2v_c6_true = w_2l2v_sm[:,None] * eft.msq_eft_over_sm(coeffs_2l2v_true, c6_space)
sigma_2l2v_c6_true = np.sum(w_2l2v_c6_true, axis=0)
# NSBI estimate
w_2l2v_c6 = w_2l2v_sm[:,None] * eft.msq_eft_over_sm(coeffs_2l2v, c6_space)
sigma_2l2v_c6 = np.sum(w_2l2v_c6, axis=0)

/ptmp/mpp/taepa/higgs-offshell-interpretation/physics/hstar/eft.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  eft_coefficients = torch.tensor(eft_coefficients)


In [10]:
nu_sm = sigma_sm * lumi
nu_c6_true = sigma_c6 * lumi
nu_c6 = sigma_c6 * lumi
nu_asimov = sigma_c6_true[i_c6_asimov]*lumi

# 4l
nu_4l_sm = sigma_4l_sm * lumi
nu_4l_c6_true = sigma_4l_c6 * lumi
nu_4l_c6 = sigma_4l_c6 * lumi
nu_4l_asimov = sigma_4l_c6_true[i_c6_asimov]*lumi

# 2l2v
nu_2l2v_sm = sigma_2l2v_sm * lumi
nu_2l2v_c6_true = sigma_2l2v_c6 * lumi
nu_2l2v_c6 = sigma_2l2v_c6 * lumi
nu_2l2v_asimov = sigma_2l2v_c6_true[i_c6_asimov]*lumi

In [11]:
t_4l_rate_true = -2 * nu_4l_asimov * (np.log(nu_4l_c6_true) - np.log(nu_4l_sm)) + 2 * (nu_4l_c6_true - nu_4l_sm) 
t_4l_rate = -2 * nu_4l_asimov * (np.log(nu_4l_c6) - np.log(nu_4l_sm)) + 2 * (nu_4l_c6 - nu_4l_sm) 

t_2l2v_rate_true = -2 * nu_2l2v_asimov * (np.log(nu_2l2v_c6_true) - np.log(nu_2l2v_sm)) + 2 * (nu_2l2v_c6_true - nu_2l2v_sm) 
t_2l2v_rate = -2 * nu_2l2v_asimov * (np.log(nu_2l2v_c6) - np.log(nu_2l2v_sm)) + 2 * (nu_2l2v_c6 - nu_2l2v_sm) 

t_rate_true = t_4l_rate_true + t_2l2v_rate_true
t_rate = t_4l_rate + t_2l2v_rate

In [12]:
pratio_4l_true = np.divide((w_4l_c6_true / sigma_4l_c6_true), (w_4l_sm[:,None] / sigma_4l_sm), where = (w_4l_sm[:,None] / sigma_4l_sm) != 0)
pratio_4l = np.divide((w_4l_c6 / sigma_4l_c6), (w_4l_sm[:,None] / sigma_4l_sm), where = (w_4l_sm[:,None] / sigma_4l_sm) != 0)

pratio_2l2v_true = np.divide((w_2l2v_c6_true / sigma_2l2v_c6_true), (w_2l2v_sm[:,None] / sigma_2l2v_sm), where = (w_2l2v_sm[:,None] / sigma_2l2v_sm) != 0)
pratio_2l2v = np.divide((w_2l2v_c6 / sigma_2l2v_c6), (w_2l2v_sm[:,None] / sigma_2l2v_sm), where = (w_2l2v_sm[:,None] / sigma_2l2v_sm) != 0)

In [13]:
t_4l_shape_true = - 2 * np.sum(lumi * w_4l_sm[:,None] * np.log(pratio_4l_true) , axis=0)
t_4l_shape = - 2 * np.sum(lumi * w_4l_sm[:,None] * np.log(pratio_4l) , axis=0)

t_2l2v_shape_true = - 2 * np.sum(lumi * w_2l2v_sm[:,None] * np.log(pratio_2l2v_true) , axis=0)
t_2l2v_shape = - 2 * np.sum(lumi * w_2l2v_sm[:,None] * np.log(pratio_2l2v) , axis=0)

t_shape_true = t_4l_shape_true + t_2l2v_shape_true
t_shape = t_4l_shape + t_2l2v_shape

In [14]:
t_4l_true = t_4l_rate_true + t_4l_shape_true
t_4l = t_4l_rate + t_4l_shape

t_2l2v_true = t_2l2v_rate_true + t_2l2v_shape_true
t_2l2v = t_2l2v_rate + t_2l2v_shape

t_true = t_rate_true + t_shape_true
t = t_rate + t_shape

In [24]:
# Plot
fig, (ax_4l, ax_2l2v, ax_zz) = plt.subplots(3,1, gridspec_kw={'height_ratios': [1,1,1]}, figsize=(5,8), sharex=True, sharey=True)

ax_4l.plot(c6_space * c6_to_cH, t_4l_true, linestyle=':', color='blue', linewidth=lw)
ax_4l.plot(c6_space * c6_to_cH, t_4l_rate, linestyle='--', color='cornflowerblue', linewidth=lw)
ax_4l.plot(c6_space * c6_to_cH, t_4l, linestyle='-', color='cornflowerblue', linewidth=lw)

from matplotlib.lines import Line2D
custom_lines = [
    Line2D([0], [0], color='black', linestyle=':', linewidth=lw, label=r'$\textrm{Parton-level}$'),
    Line2D([0], [0], color='grey', linestyle='--', linewidth=lw, label=r'$\textrm{Rate}$'),
    Line2D([0], [0], color='grey', linestyle='-', linewidth=lw, label=r'$\textrm{NSBI}$'),
]
ax_4l.legend(frameon=False, handles=custom_lines, loc='upper right', fontsize=12)

ax_4l.axhline(1, color='lightgrey', linestyle='--', linewidth=lw)
ax_4l.axhline(4, color='lightgrey', linestyle='--', linewidth=lw)
ax_4l.text(x=96, y=1.2, s='$1\\sigma$', color='grey', fontsize=12, va='bottom', ha='right')
ax_4l.text(x=96, y=4.2, s='$2\\sigma$', color='grey', fontsize=12, va='bottom', ha='right')

ax_4l.yaxis.set_ticks([0,1,4,9])

ax_4l.yaxis.set_tick_params(labelsize=12)
ax_4l.xaxis.set_tick_params(labelsize=12)

plt.text(0.96 ,0.32, '$gg \\to (h^{\\ast}\\to)\\, ZZ \\to 4\\ell$', transform=ax_4l.transAxes, ha='right', va='center', fontsize=12, color='blue')
plt.text(0.96 ,0.24, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax_4l.transAxes, ha='right', va='center', fontsize=12)

# ax_4l.legend(frameon=False, fontsize=12, loc='upper right')

ax_2l2v.plot(c6_space * c6_to_cH, t_2l2v_true, label='$\\textrm{Parton-level}$', linestyle=':', color='red', linewidth=lw)
ax_2l2v.plot(c6_space * c6_to_cH, t_2l2v_rate, label='$\\textrm{Rate}$', linestyle='--', color='salmon', linewidth=lw)
ax_2l2v.plot(c6_space * c6_to_cH, t_2l2v, label='$\\textrm{NSBI}$', linestyle='-', color='salmon', linewidth=lw)

ax_2l2v.axhline(1, color='lightgrey', linestyle='--', linewidth=lw)
ax_2l2v.axhline(4, color='lightgrey', linestyle='--', linewidth=lw)
ax_2l2v.text(x=96, y=1.2, s='$1\\sigma$', color='grey', fontsize=12, va='bottom', ha='right')
ax_2l2v.text(x=96, y=4.2, s='$2\\sigma$', color='grey', fontsize=12, va='bottom', ha='right')

ax_2l2v.set_ylabel('$-2\\Delta\\ln\\lambda$',fontsize=15)
ax_2l2v.set_ylim(0, 10)

ax_2l2v.yaxis.set_ticks([0,1,4,9])

ax_2l2v.xaxis.set_tick_params(which='both', labeltop=False, top=True)
ax_2l2v.yaxis.set_tick_params(labelsize=12)
ax_2l2v.xaxis.set_tick_params(labelsize=12)

plt.text(0.96 ,0.32, '$gg \\to (h^{\\ast}\\to)\\, ZZ \\to 2\\ell 2\\nu$', transform=ax_2l2v.transAxes, ha='right', va='center', fontsize=12, color='red')
plt.text(0.96 ,0.24, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax_2l2v.transAxes, ha='right', va='center', fontsize=12)

# ax_2l2v.legend(frameon=False, fontsize=12, loc='upper right')

ax_zz.plot(c6_space * c6_to_cH, t_true, label='$\\textrm{Parton-level}$', linestyle=':', color='black', linewidth=lw)
ax_zz.plot(c6_space * c6_to_cH, t_rate, label='$\\textrm{Rate}$', linestyle='--', color='grey', linewidth=lw)
ax_zz.plot(c6_space * c6_to_cH, t, label='$\\textrm{NSBI}$', linestyle='-', color='grey', linewidth=lw)

ax_zz.axhline(1, color='lightgrey', linestyle='--', linewidth=lw)
ax_zz.axhline(4, color='lightgrey', linestyle='--', linewidth=lw)
ax_zz.text(x=96, y=1.2, s='$1\\sigma$', color='grey', fontsize=12, va='bottom', ha='right')
ax_zz.text(x=96, y=4.2, s='$2\\sigma$', color='grey', fontsize=12, va='bottom', ha='right')

# Aesthetics
ax_zz.set_xlabel('$C_H$',fontsize=15)
ax_zz.set_xlim(-50, 100)

ax_zz.yaxis.set_ticks([0,1,4,9])
ax_zz.xaxis.set_ticks([-50, -25, -10, 0, 10, 25, 50, 100])
from matplotlib.ticker import FixedLocator
minor_ticks = [-25, -45, -40, -35, -30, -20, -15, -5, 5, 15, 20, 30, 35, 40, 45]
ax_zz.xaxis.set_minor_locator(FixedLocator(minor_ticks))

ax_zz.xaxis.set_tick_params(which='both', labeltop=False, top=True)
ax_zz.yaxis.set_tick_params(labelsize=12)
ax_zz.xaxis.set_tick_params(labelsize=12)

plt.text(0.96 ,0.32, '$gg \\to (h^{\\ast}\\to)\\, ZZ$', transform=ax_zz.transAxes, ha='right', va='center', fontsize=12, color='black')
plt.text(0.96 ,0.24, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax_zz.transAxes, ha='right', va='center', fontsize=12)

# ax_zz.legend(frameon=False, fontsize=12, loc='upper right')

plt.tight_layout()
plt.subplots_adjust(hspace=0.0)

plt.savefig('nll_gg_zz.pdf', bbox_inches='tight')

In [20]:
cH_space = c6_space * c6_to_cH

print(np.min(cH_space[np.where(t <= 1.0)]), np.max(cH_space[np.where(t <= 1.0)]))
print(np.min(cH_space[np.where(t <= 4.0)]), np.max(cH_space[np.where(t <= 4.0)]))

-13.328094358169714 7.996856614901829
-26.65618871633943 13.861218132496504
